In [2]:
"""
Ablation: Autoencoder Only (No Multi-Task Learning)
====================================================
Architecture:  Tensor fusion + BiLSTM + reconstruction decoders
Training:      Reconstruction loss ONLY (MSE on img/tab/time encoder outputs)
Ablation role: Tests whether unsupervised reconstruction learns useful
               representations without any survival supervision.

What differs from Thesis:
- Loss: reconstruction ONLY — no Cox, no MMSE head
- Adds decoder heads for img, tab, time reconstruction
What is kept identical to Thesis:
- Same engineer_features(), same TEMPORAL_FEATURES
- Same tensor fusion, same BiLSTM
- Same scaler-on-train-only fix, same apply_scaler(), same PTID guards
- dropout=0.4, GELU activations, Xavier init matched to MSTE-JM
- Same export format
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================
CONFIG = {
    'latent_dim': 128,
    'img_out_dim': 256,
    'tab_out_dim': 64,
    'lstm_hidden': 128,
    'lstm_layers': 2,
    'dropout': 0.4,       # matched to MSTE-JM
    'lr': 5e-4,
    'weight_decay': 1e-4,
    'epochs': 100,
    'batch_size': 16,
    'accumulation_steps': 1,
    'max_seq_len': 10,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

# ============================================================================
# FEATURE DEFINITIONS — identical to thesis
# ============================================================================
STATIC_FEATURES = [
    'age_bl', 'PTGENDER_encoded', 'PTEDUCAT', 'PTETHCAT_encoded',
    'PTRACCAT_encoded', 'PTMARRY_encoded'
]

TEMPORAL_FEATURES = [
    'time_from_baseline', 'AGE', 'age_since_bl',
    'mmse_slope', 'visit_number', 'MMSE', 'ADAS13',
    'mmse_variability', 'adas_mmse_discordance', 'mmse_acceleration'
]

# ============================================================================
# FEATURE ENGINEERING — identical to thesis (causal, no future leakage)
# ============================================================================
def engineer_features(df):
    """Engineer temporal features using only past data at each time step."""
    df = df.sort_values("Years_bl").reset_index(drop=True)

    df["mmse_slope"] = 0.0
    df["mmse_acceleration"] = 0.0
    df["mmse_variability"] = 0.0
    df["adas_mmse_discordance"] = 0.0

    for t in range(len(df)):
        df_past = df.iloc[:t + 1]

        if len(df_past) >= 2:
            dy = df_past["MMSE"].iloc[-1] - df_past["MMSE"].iloc[-2]
            dt = df_past["Years_bl"].iloc[-1] - df_past["Years_bl"].iloc[-2]
            df.loc[t, "mmse_slope"] = dy / dt if dt != 0 else 0.0

        if t >= 2:
            df.loc[t, "mmse_acceleration"] = df.loc[t, "mmse_slope"] - df.loc[t - 1, "mmse_slope"]

        if len(df_past) >= 2:
            df.loc[t, "mmse_variability"] = df_past["MMSE"].std()

        df.loc[t, "adas_mmse_discordance"] = abs(
            df_past["ADAS13"].iloc[-1] - df_past["MMSE"].iloc[-1]
        )

    df = df.ffill()
    df = df.fillna(0)
    return df

# ============================================================================
# DATASET — raw, unscaled. Scaler applied externally after split.
# ============================================================================
class SequenceDataset(Dataset):
    def __init__(self, manifest, valid_patients, transform=None, max_seq_len=10):
        self.transform = transform
        self.max_seq_len = max_seq_len
        self.sequences = []
        self.scaler = None  # set externally after split

        manifest = manifest.copy()
        manifest["path"] = manifest["path"].str.replace("\\", "/", regex=False)
        manifest["path"] = "./AD_Multimodal/TFN_AD/" + manifest["path"]

        skipped_not_valid = 0
        skipped_errors = 0
        processed = 0

        for ptid in manifest["PTID"].unique():
            if ptid not in valid_patients:
                skipped_not_valid += 1
                continue

            try:
                patient_rows = manifest[manifest["PTID"] == ptid]
                if len(patient_rows) == 0:
                    continue

                df = pd.read_pickle(patient_rows.iloc[0]["path"])
                df = engineer_features(df)

                dx_seq = df["DX"].tolist()
                if "MCI" not in dx_seq:
                    continue

                mci_idx = dx_seq.index("MCI")
                ad_idx = next(
                    (i for i, x in enumerate(dx_seq[mci_idx + 1:], start=mci_idx + 1)
                     if x in ["AD", "Dementia"]),
                    -1
                )

                if ad_idx != -1:
                    time_to_event = (
                        df["Years_bl"].iloc[ad_idx] - df["Years_bl"].iloc[mci_idx]
                    )
                    event = 1
                else:
                    time_to_event = (
                        df["Years_bl"].iloc[-1] - df["Years_bl"].iloc[mci_idx]
                    )
                    event = 0

                imgs, tabs, times, mmse_vals = [], [], [], []
                valid_visits = 0

                for _, visit in df.iterrows():
                    image_path = visit["image_path"].replace(
                        "/home/mason/ADNI_Dataset/",
                        "./AD_Multimodal/ADNI_Dataset/"
                    )
                    if not os.path.exists(image_path):
                        continue

                    img = Image.open(image_path).convert("RGB")
                    if self.transform:
                        img = self.transform(img)
                    imgs.append(img)

                    feature_cols = TEMPORAL_FEATURES + STATIC_FEATURES
                    row_vals = []
                    for col in feature_cols:
                        row_vals.append(float(visit[col]) if col in visit.index else 0.0)
                    tabs.append(np.array(row_vals, dtype=np.float32))

                    times.append(float(visit["Years_bl"]))
                    mmse_vals.append(float(visit["MMSE"]))
                    valid_visits += 1

                    if valid_visits >= self.max_seq_len:
                        break

                if valid_visits < 2:
                    continue

                pad_len = self.max_seq_len - valid_visits
                for _ in range(pad_len):
                    imgs.append(torch.zeros_like(imgs[-1]))
                    tabs.append(np.zeros_like(tabs[-1]))
                    times.append(times[-1])
                    mmse_vals.append(np.nan)

                self.sequences.append({
                    "ptid": ptid,
                    "imgs": torch.stack(imgs),
                    "tabs": np.array(tabs, dtype=np.float32),  # RAW, unscaled
                    "times": np.array(times, dtype=np.float32),
                    "mmse": np.array(mmse_vals, dtype=np.float32),
                    "seq_len": valid_visits,
                    "time_to_event": float(time_to_event),
                    "event": int(event),
                })
                processed += 1

            except Exception as e:
                skipped_errors += 1
                print(f"  ⚠️  Skipped patient {ptid}: {type(e).__name__}: {e}")
                continue

        print(f"  Processed              : {processed} valid patients")
        print(f"  Skipped (not in valid) : {skipped_not_valid}")
        print(f"  Skipped (errors)       : {skipped_errors}")

    def apply_scaler(self, scaler):
        self.scaler = scaler
        for seq in self.sequences:
            real_len = seq['seq_len']
            seq['tabs'][:real_len] = scaler.transform(seq['tabs'][:real_len])
            if real_len < len(seq['tabs']):
                seq['tabs'][real_len:] = 0.0

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return (
            seq['imgs'],
            seq['tabs'],
            seq['times'],
            seq['mmse'],
            seq['seq_len'],
            seq['time_to_event'],
            seq['event'],
            seq['ptid'],
        )

# ============================================================================
# WEIGHT INITIALISATION — Xavier uniform, matched to MSTE-JM
# ============================================================================
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

# ============================================================================
# MODEL — tensor fusion + BiLSTM + reconstruction decoders; no survival head
#         GELU throughout, dropout=0.4, matched to MSTE-JM
# ============================================================================
class TensorFusion(nn.Module):
    def __init__(self, v_dim, d_dim, t_dim, dropout=0.1):
        super().__init__()
        self.output_dim = (v_dim + 1) * (d_dim + 1) * (t_dim + 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, v, d, t):
        bs = v.shape[0]
        v1 = torch.cat([v, torch.ones(bs, 1, device=v.device)], dim=1)
        d1 = torch.cat([d, torch.ones(bs, 1, device=d.device)], dim=1)
        t1 = torch.cat([t, torch.ones(bs, 1, device=t.device)], dim=1)
        fusion = torch.einsum('bi,bj,bk->bijk', v1, d1, t1).view(bs, -1)
        return self.dropout(fusion)


class AttentionImageEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for param in list(base.parameters())[:-20]:
            param.requires_grad = False
        self.features = nn.Sequential(*list(base.children())[:-2])
        self.attention = nn.Sequential(
            nn.Conv2d(512, 256, 1), nn.BatchNorm2d(256), nn.GELU(),   # GELU matched to MSTE-JM
            nn.Conv2d(256, 1, 1), nn.Sigmoid()
        )
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(512, out_dim)

    def forward(self, x):
        feats = self.features(x)
        feats = feats * self.attention(feats)
        return self.proj(self.global_pool(feats).view(x.size(0), -1))


class AutoencoderOnlyModel(nn.Module):
    """
    KEY DIFFERENCE vs thesis: trained with reconstruction loss only.
    Encoder is identical to thesis (tensor fusion + BiLSTM + latent proj).
    Hyperparameters (dropout=0.4, GELU, Xavier init) matched to MSTE-JM.
    No survival head, no MMSE head.
    """
    def __init__(self, tab_dim, config):
        super().__init__()
        self.config = config

        self.img_encoder = AttentionImageEncoder(out_dim=config['img_out_dim'])
        self.tab_encoder = nn.Sequential(
            nn.Linear(tab_dim, 128), nn.BatchNorm1d(128), nn.GELU(),   # GELU
            nn.Dropout(config['dropout']),
            nn.Linear(128, config['tab_out_dim']), nn.GELU()            # GELU
        )
        self.time_encoder = nn.Sequential(nn.Linear(1, 16), nn.GELU()) # GELU

        self.fusion = TensorFusion(
            v_dim=config['img_out_dim'],
            d_dim=config['tab_out_dim'],
            t_dim=16,
            dropout=config['dropout']
        )
        fusion_dim = (config['img_out_dim'] + 1) * (config['tab_out_dim'] + 1) * 17
        self.fusion_proj = nn.Sequential(
            nn.Linear(fusion_dim, config['latent_dim']),
            nn.BatchNorm1d(config['latent_dim']), nn.GELU()             # GELU
        )
        self.lstm = nn.LSTM(
            input_size=config['latent_dim'],
            hidden_size=config['lstm_hidden'],
            num_layers=config['lstm_layers'],
            batch_first=True,
            dropout=config['dropout'] if config['lstm_layers'] > 1 else 0,
            bidirectional=True
        )
        self.temporal_proj = nn.Sequential(
            nn.Linear(config['lstm_hidden'] * 2, config['latent_dim']), nn.GELU()  # GELU
        )

        # Decoder heads — reconstruct encoder intermediate representations
        self.img_decoder = nn.Sequential(
            nn.Linear(config['latent_dim'], 512), nn.GELU(),            # GELU
            nn.Dropout(config['dropout']),
            nn.Linear(512, config['img_out_dim'])
        )
        self.tab_decoder = nn.Sequential(
            nn.Linear(config['latent_dim'], 128), nn.GELU(),            # GELU
            nn.Dropout(config['dropout']),
            nn.Linear(128, config['tab_out_dim'])
        )
        self.time_decoder = nn.Sequential(
            nn.Linear(config['latent_dim'], 32), nn.GELU(),             # GELU
            nn.Dropout(config['dropout']),
            nn.Linear(32, 16)
        )

    def encode_visit(self, img, tab, time):
        """Per-timestep encoding — called once per time step in forward()."""
        v = self.img_encoder(img)
        d = self.tab_encoder(tab)
        t = self.time_encoder(time.unsqueeze(1))
        v = F.normalize(v, dim=1)
        d = F.normalize(d, dim=1)
        t = F.normalize(t, dim=1)

        z = self.fusion(v, d, t)
        z = z.view(z.size(0), -1)
        z = self.fusion_proj(z)
        return z, v, d, t

    def forward(self, img_seq, tab_seq, time_seq, seq_lengths):
        bs, seq_len = img_seq.shape[:2]

        z_list, v_list, d_list, t_list = [], [], [], []
        for step in range(seq_len):
            z_t, v_t, d_t, t_t = self.encode_visit(
                img_seq[:, step], tab_seq[:, step], time_seq[:, step]
            )
            z_list.append(z_t)
            v_list.append(v_t)
            d_list.append(d_t)
            t_list.append(t_t)

        z_seq = torch.stack(z_list, dim=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            z_seq, seq_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        z_final = self.temporal_proj(h_final)

        v_recon = self.img_decoder(z_final)
        d_recon = self.tab_decoder(z_final)
        t_recon = self.time_decoder(z_final)

        return z_final, v_recon, d_recon, t_recon, v_list, d_list, t_list

# ============================================================================
# LOSS — reconstruction only
# ============================================================================
def reconstruction_loss(z_final, v_recon, d_recon, t_recon,
                        v_list, d_list, t_list, seq_lengths):
    loss = torch.tensor(0.0, device=z_final.device)
    B = z_final.shape[0]
    for i in range(B):
        last = seq_lengths[i] - 1
        loss += F.mse_loss(v_recon[i], v_list[last][i].detach())
        loss += F.mse_loss(d_recon[i], d_list[last][i].detach())
        loss += F.mse_loss(t_recon[i], t_list[last][i].detach())
    return loss / B

# ============================================================================
# TRAINING & VALIDATION
# ============================================================================
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        imgs, tabs, times, mmse, seq_lens, t_event, event, _ = batch

        imgs     = imgs.to(device)
        tabs     = torch.FloatTensor(tabs).to(device)
        times    = torch.FloatTensor(times).to(device)
        seq_lens = torch.LongTensor(seq_lens)

        z_final, v_recon, d_recon, t_recon, v_list, d_list, t_list = model(
            imgs, tabs, times, seq_lens
        )
        loss = reconstruction_loss(
            z_final, v_recon, d_recon, t_recon, v_list, d_list, t_list, seq_lens
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return {'total': total_loss / len(loader)}


def validate(model, loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, mmse, seq_lens, t_event, event, _ = batch

            imgs     = imgs.to(device)
            tabs     = torch.FloatTensor(tabs).to(device)
            times    = torch.FloatTensor(times).to(device)
            seq_lens = torch.LongTensor(seq_lens)

            z_final, v_recon, d_recon, t_recon, v_list, d_list, t_list = model(
                imgs, tabs, times, seq_lens
            )
            loss = reconstruction_loss(
                z_final, v_recon, d_recon, t_recon, v_list, d_list, t_list, seq_lens
            )
            total_loss += loss.item()

    return total_loss / len(loader)

# ============================================================================
# EXPORT
# ============================================================================
def export_latent_features(model, loader, device, output_path):
    model.eval()
    rows = []
    BASELINE_FEATURES = ['AGE', 'PTGENDER', 'PTEDUCAT', 'ADAS13']
    tab_feature_names = TEMPORAL_FEATURES + STATIC_FEATURES

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, mmse, seq_lens, t_event, event, ptids = batch
            imgs     = imgs.to(device)
            tabs     = tabs.to(device)
            times    = times.to(device)
            seq_lens = seq_lens.cpu().long()

            for i in range(len(ptids)):
                slen = seq_lens[i].item()
                for t in range(slen):
                    z_visit, _, _, _ = model.encode_visit(
                        imgs[i:i + 1, t], tabs[i:i + 1, t], times[i:i + 1, t]
                    )
                    base_dataset = (
                        loader.dataset.dataset
                        if hasattr(loader.dataset, 'dataset')
                        else loader.dataset
                    )
                    tab_vals_unscaled = base_dataset.scaler.inverse_transform(
                        tabs[i, t].cpu().numpy().reshape(1, -1)
                    )[0]

                    row = {
                        "PTID": ptids[i],
                        "Years_bl": float(times[i, t].cpu()),
                        "MMSE": float(mmse[i, t]),
                        "time_to_event": float(t_event[i]),
                        "event": int(event[i]),
                    }
                    for feat in BASELINE_FEATURES:
                        feat_encoded = feat + '_encoded' if feat == 'PTGENDER' else feat
                        if feat_encoded in tab_feature_names:
                            idx = tab_feature_names.index(feat_encoded)
                            row[feat] = float(tab_vals_unscaled[idx])

                    for k, val in enumerate(z_visit[0].cpu().numpy()):
                        row[f"z_{k}"] = float(val)

                    rows.append(row)

    df = pd.DataFrame(rows).sort_values(["PTID", "Years_bl"])
    df.to_csv(output_path, index=False)
    print(f"\n✓ Exported to {output_path}")
    print(f"  Patients        : {df['PTID'].nunique()}")
    print(f"  Visits          : {len(df)}")
    print(f"  Latent features : {len([c for c in df.columns if c.startswith('z_')])}")
    return df


def export_patient_level_ae_only(model, loader, device, output_path):
    model.eval()
    rows = []

    with torch.no_grad():
        for batch in loader:
            imgs, tabs, times, mmse, seq_lens, t_event, event, ptids = batch

            imgs       = imgs.to(device)
            tabs       = tabs.to(device)
            times_dev  = times.to(device)
            times_np   = times.numpy()
            seq_lens_t = torch.LongTensor(seq_lens)

            z_final, _, _, _, _, _, _ = model(imgs, tabs, times_dev, seq_lens_t)

            bs, seq_len = imgs.shape[:2]
            z_visits = []
            for t in range(seq_len):
                z_t, _, _, _ = model.encode_visit(imgs[:, t], tabs[:, t], times_dev[:, t])
                z_visits.append(z_t.cpu().numpy())
            z_visits = np.array(z_visits)

            for i in range(len(ptids)):
                slen = int(seq_lens[i])
                row = {
                    "PTID":          ptids[i],
                    "time_to_event": float(t_event[i]),
                    "event":         int(event[i]),
                    "risk_score":    0.0,   # AE-only has no survival head
                    "MMSE":          float(mmse[i, slen - 1]),
                    "Years_bl":      float(times_np[i, slen - 1]),
                    "n_visits":      slen,
                }

                for k, val in enumerate(z_final[i].cpu().numpy()):
                    row[f"z_final_{k}"] = float(val)

                real_z = z_visits[:slen, i, :]
                dt     = times_np[i, :slen] - times_np[i, 0]

                if slen >= 2 and dt[-1] > 0:
                    for k in range(real_z.shape[1]):
                        slope = np.cov(dt, real_z[:, k])[0, 1] / (np.var(dt) + 1e-8)
                        row[f"z_slope_{k}"] = float(slope)
                else:
                    for k in range(real_z.shape[1]):
                        row[f"z_slope_{k}"] = 0.0

                rows.append(row)

    df = pd.DataFrame(rows).sort_values("PTID").reset_index(drop=True)
    df.to_csv(output_path, index=False)
    n_zf = len([c for c in df.columns if c.startswith("z_final_")])
    n_zs = len([c for c in df.columns if c.startswith("z_slope_")])
    print(f"\n✓ AE-Only patient-level export: {len(df)} patients → {output_path}")
    print(f"  z_final: {n_zf}  z_slope: {n_zs}")
    print(f"  NOTE: risk_score=0.0 (no survival head — downstream Cox model provides ranking)")
    return df

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("=" * 80)
    print("ABLATION: Autoencoder Only (No Multi-Task Learning)")
    print("=" * 80)

    device = CONFIG['device']
    print(f"\nDevice: {device}")

    print("\nLoading data...")
    manifest = pd.read_csv("./AD_Multimodal/TFN_AD/AD_Patient_Manifest.csv")

    print("\nLoading valid patient list...")
    with open('VALID_PATIENTS.pkl', 'rb') as f:
        VALID_PATIENTS = pickle.load(f)

    VALID_PATIENTS = set(str(p) for p in VALID_PATIENTS)
    manifest["PTID"] = manifest["PTID"].astype(str)

    print(f"Valid patients in pkl : {len(VALID_PATIENTS)}")
    overlap = set(manifest["PTID"].unique()) & VALID_PATIENTS
    print(f"Overlap with manifest : {len(overlap)}")
    if len(overlap) == 0:
        raise RuntimeError(
            "No PTID overlap between manifest and VALID_PATIENTS.pkl — "
            "check for type mismatches (int vs str) or ID format differences."
        )

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    print("\nCreating dataset...")
    dataset = SequenceDataset(manifest, VALID_PATIENTS, transform,
                              max_seq_len=CONFIG['max_seq_len'])

    print(f"\nTotal sequences: {len(dataset)}")
    if len(dataset) == 0:
        raise RuntimeError("Dataset is empty after processing.")

    with open('TRAIN_PATIENTS.pkl', 'rb') as f:
        TRAIN_PATIENTS = pickle.load(f)
    with open('VAL_PATIENTS.pkl', 'rb') as f:
        VAL_PATIENTS = pickle.load(f)

    train_indices = [i for i, s in enumerate(dataset.sequences) if s['ptid'] in TRAIN_PATIENTS]
    val_indices   = [i for i, s in enumerate(dataset.sequences) if s['ptid'] in VAL_PATIENTS]

    train_dataset = torch.utils.data.Subset(dataset, train_indices)
    val_dataset   = torch.utils.data.Subset(dataset, val_indices)

    print(f"Train: {len(train_indices)}  |  Val (test): {len(val_indices)}")

    train_tabs = np.vstack([
        dataset.sequences[i]['tabs'][:dataset.sequences[i]['seq_len']]
        for i in train_indices
    ])
    scaler = StandardScaler()
    scaler.fit(train_tabs)
    dataset.apply_scaler(scaler)
    print("Scaler fitted on train split only — no val leakage ✓")

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                              shuffle=True, num_workers=0)
    val_loader   = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                              shuffle=False, num_workers=0)

    tab_dim = dataset.sequences[0]['tabs'].shape[1]
    print(f"\nTab feature dimension : {tab_dim}")
    print("Initializing model...")
    model = AutoencoderOnlyModel(tab_dim, CONFIG).to(device)
    model.apply(init_weights)   # Xavier init matched to MSTE-JM
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=CONFIG['lr'],
                                  weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=10
    )

    print("\n" + "=" * 80)
    print("TRAINING")
    print("=" * 80)

    best_loss = float('inf')
    patience_ctr = 0

    for epoch in range(CONFIG['epochs']):
        losses = train_epoch(model, train_loader, optimizer, device)
        val_loss = validate(model, val_loader, device)
        scheduler.step(val_loss)

        print(f"\nEpoch {epoch + 1}/{CONFIG['epochs']}")
        print(f"  Train recon loss : {losses['total']:.4f}")
        print(f"  Val   recon loss : {val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'loss': best_loss,
            }, 'best_model_ae_only.pth')
            print(f"  ✓ New best: {best_loss:.4f}")
            patience_ctr = 0
        else:
            patience_ctr += 1

        if patience_ctr >= 15:
            print("\n⚠  Early stopping triggered.")
            break

    print("\n" + "=" * 80)
    print("EXPORTING")
    print("=" * 80)

    checkpoint = torch.load('best_model_ae_only.pth', weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nLoaded best model (epoch {checkpoint['epoch'] + 1}, "
          f"loss {checkpoint['loss']:.4f})")

    full_loader_export = DataLoader(
        torch.utils.data.Subset(dataset, list(range(len(dataset)))),
        batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0
    )

    df_long = export_latent_features(model, full_loader_export, device, "latent_ae_only.csv")
    df_long['split'] = df_long['PTID'].apply(
        lambda p: 'train' if str(p) in TRAIN_PATIENTS else 'val'
    )
    df_long.to_csv("latent_ae_only.csv", index=False)

    df_pl = export_patient_level_ae_only(model, full_loader_export, device, "ae_only_patient_level.csv")
    df_pl['split'] = df_pl['PTID'].apply(
        lambda p: 'train' if str(p) in TRAIN_PATIENTS else 'val'
    )
    df_pl.to_csv("ae_only_patient_level.csv", index=False)

    print(f"\nBest reconstruction loss : {best_loss:.4f}")
    print("=" * 80)


if __name__ == "__main__":
    main()

ABLATION: Autoencoder Only (No Multi-Task Learning)

Device: cuda

Loading data...

Loading valid patient list...
Valid patients in pkl : 382
Overlap with manifest : 382

Creating dataset...
  Processed              : 161 valid patients
  Skipped (not in valid) : 0
  Skipped (errors)       : 0

Total sequences: 161
Train: 128  |  Val (test): 33
Scaler fitted on train split only — no val leakage ✓

Tab feature dimension : 16
Initializing model...
Parameters: 48,720,273

TRAINING

Epoch 1/100
  Train recon loss : 0.0669
  Val   recon loss : 0.0390
  ✓ New best: 0.0390

Epoch 2/100
  Train recon loss : 0.0415
  Val   recon loss : 0.0265
  ✓ New best: 0.0265

Epoch 3/100
  Train recon loss : 0.0350
  Val   recon loss : 0.0180
  ✓ New best: 0.0180

Epoch 4/100
  Train recon loss : 0.0314
  Val   recon loss : 0.0166
  ✓ New best: 0.0166

Epoch 5/100
  Train recon loss : 0.0283
  Val   recon loss : 0.0142
  ✓ New best: 0.0142

Epoch 6/100
  Train recon loss : 0.0263
  Val   recon loss : 0.013